<a href="https://colab.research.google.com/github/GanyaGit/GreenInfer/blob/main/greeninfer_week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests pandas plotly

In [ ]:
import requests
import pandas as pd
from datetime import datetime


# GreenInfer — Week 1
# Electricity Maps API client


API_KEY = "api_key"

HEADERS = {"auth-token": API_KEY}
BASE_URL = "https://api.electricitymap.org/v3"

def get_carbon_intensity(zone: str) -> dict:
    """
    Get real-time carbon intensity (gCO2eq/kWh) for a region.
    zone examples: "CN" (China), "US-CAL-CISO" (California), "DE" (Germany)
    """
    url = f"{BASE_URL}/carbon-intensity/latest?zone={zone}"
    response = requests.get(url, headers=HEADERS)

    if response.status_code == 200:
        data = response.json()
        return {
            "zone": zone,
            "carbon_intensity_gco2_per_kwh": data["carbonIntensity"],
            "timestamp": data["datetime"],
            "fetched_at": datetime.utcnow().isoformat()
        }
    else:
        return {"error": response.status_code, "zone": zone}


def get_power_breakdown(zone: str) -> dict:
    """
    Get how electricity is being generated right now — solar, coal, wind etc.
    """
    url = f"{BASE_URL}/power-breakdown/latest?zone={zone}"
    response = requests.get(url, headers=HEADERS)

    if response.status_code == 200:
        data = response.json()
        return {
            "zone": zone,
            "renewable_percentage": data.get("renewablePercentage"),
            "fossil_fuel_percentage": data.get("fossilFuelPercentage"),
            "power_sources": data.get("powerProductionBreakdown", {})
        }
    else:
        return {"error": response.status_code, "zone": zone}


# Test it with 3 major data center regions
zones = ["CN", "US-CAL-CISO", "DE"]

results = []
for zone in zones:
    ci = get_carbon_intensity(zone)
    pb = get_power_breakdown(zone)
    results.append({**ci, **pb})
    print(f"✅ {zone} — {ci.get('carbon_intensity_gco2_per_kwh')} gCO2/kWh")

# Save to DataFrame
df = pd.DataFrame(results)
print("\nThe first dataset")
print(df[["zone", "carbon_intensity_gco2_per_kwh",
          "renewable_percentage", "fossil_fuel_percentage"]])

# Save to CSV — this goes into HDFS later
df.to_csv("electricity_data_day1.csv", index=False)
print("\n✅ Saved to electricity_data_day1.csv")